<a href="https://colab.research.google.com/github/Tomax47/Uni-Crawler-Tasks/blob/main/crawler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import zipfile
from google.colab import files
import time

In [6]:
SEED_URL = "https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA"
MAX_PAGES = 100
OUTPUT_DIR = "pages"
INDEX_FILE = "index.txt"
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

In [7]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def is_valid_url(url, base_domain):
    parsed = urlparse(url)
    return bool(parsed.netloc) and parsed.netloc == base_domain

def crawl(seed_url, max_pages):
    visited = set()
    queue = [seed_url]
    count = 0
    base_domain = urlparse(seed_url).netloc

    with open(INDEX_FILE, "w", encoding="utf-8") as index_f:
        while queue and count < max_pages:
            url = queue.pop(0)

            if url in visited or "#" in url:
                continue

            try:
                response = requests.get(url, headers=HEADERS, timeout=10)

                if response.status_code != 200:
                    print(f"Skipping {url} (Status: {response.status_code})")
                    continue

                content_type = response.headers.get('Content-Type', '')
                if 'text/html' not in content_type:
                    continue

                file_name = f"{count}.html"
                file_path = os.path.join(OUTPUT_DIR, file_name)

                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(response.text)

                index_f.write(f"{count} {url}\n")
                visited.add(url)
                print(f"[{count}] Saved: {url}")
                count += 1

                soup = BeautifulSoup(response.text, 'html.parser')
                for link in soup.find_all('a', href=True):
                    full_url = urljoin(url, link['href'])
                    full_url = full_url.split('?')[0].split('#')[0]

                    if is_valid_url(full_url, base_domain) and full_url not in visited:
                        queue.append(full_url)

                time.sleep(0.1)

            except Exception as e:
                print(f"Error crawling {url}: {e}")

    return count

In [8]:
print(f"Starting crawl at: {SEED_URL}")
total_downloaded = crawl(SEED_URL, MAX_PAGES)
print(f"\nFinished. Total pages downloaded: {total_downloaded}")

if total_downloaded > 0:
    def create_archive(zip_name, folder_path, index_path):
        with zipfile.ZipFile(zip_name, 'w') as zipf:
            zipf.write(index_path, arcname=index_path)
            for root, dirs, files_list in os.walk(folder_path):
                for file in files_list:
                    zipf.write(os.path.join(root, file),
                               arcname=os.path.join(folder_path, file))

    archive_name = "tema1_result.zip"
    create_archive(archive_name, OUTPUT_DIR, INDEX_FILE)
    print(f"Downloading {archive_name}...")
    files.download(archive_name)
else:
    print("No pages were downloaded")

Starting crawl at: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA
[0] Saved: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B9_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA
[1] Saved: https://ru.wikipedia.org/wiki/%D0%90%D0%BD%D0%B3%D0%BB%D0%B8%D0%B9%D1%81%D0%BA%D0%B8%D0%B9_%D1%8F%D0%B7%D1%8B%D0%BA
[2] Saved: https://ru.wikipedia.org/wiki/%D0%9F%D0%BE%D0%B8%D1%81%D0%BA
[3] Saved: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D1%8F
[4] Saved: https://ru.wikipedia.org/wiki/%D0%98%D0%BD%D1%84%D0%BE%D1%80%D0%BC%D0%B0%D1%86%D0%B8%D0%BE%D0%BD%D0%BD%D1%8B%D0%B5_%D0%BF%D0%BE%D1%82%D1%80%D0%B5%D0%B1%D0%BD%D0%BE%D1%81%D1%82%D0%B8
[5] Saved: https://ru.wikipedia.org/wiki/%D0%9D%D0%B0%D1%83%D0%BA%D0%B0
[6] Saved: https://ru.wikipedia.org/w/index.php
[7] Saved: https://ru.wikipedia.org/wiki/1948
[8] Saved: https:/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task-2: Tokenizaiton

In [9]:
!pip install pymorphy3 nltk

import re
import nltk
from pymorphy3 import MorphAnalyzer
from nltk.corpus import stopwords

nltk.download('stopwords')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 56.7 MB/s eta 0:00:00


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [10]:
INPUT_DIR = "pages"
TOKENS_FILE = "tokens.txt"
LEMMAS_FILE = "lemmas.txt"

morph = MorphAnalyzer()
russian_stopwords = set(stopwords.words('russian'))
EXCLUDED_POS = {'PREP', 'CONJ', 'PRCL', 'INTJ'}

In [11]:
def is_valid_word(word):
    return bool(re.fullmatch(r'[а-яё]+', word))

def extract_and_process(directory):
    unique_tokens = set()
    lemma_map = {}

    if not os.path.exists(directory):
        print(f"Error: {directory} folder not found")
        return unique_tokens, lemma_map

    for filename in sorted(os.listdir(directory)):
        if filename.endswith(".html"):
            file_path = os.path.join(directory, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f.read(), 'html.parser')

                for element in soup(["script", "style", "meta", "noscript"]):
                    element.decompose()

                text = soup.get_text().lower()
                potential_words = re.findall(r'[а-яё0-9a-z]+', text)

                for word in potential_words:
                    if not is_valid_word(word) or len(word) < 2:
                        continue

                    if word in russian_stopwords:
                        continue

                    analysis = morph.parse(word)[0]
                    pos = analysis.tag.POS
                    if pos in EXCLUDED_POS:
                        continue

                    unique_tokens.add(word)

                    lemma = analysis.normal_form
                    if lemma not in lemma_map:
                        lemma_map[lemma] = set()
                    lemma_map[lemma].add(word)

    return sorted(list(unique_tokens)), lemma_map

In [12]:
print("Processing...")
tokens, lemmas = extract_and_process(INPUT_DIR)

with open(TOKENS_FILE, "w", encoding="utf-8") as f:
    for token in tokens:
        f.write(f"{token}\n")

with open(LEMMAS_FILE, "w", encoding="utf-8") as f:
    for lemma, token_set in sorted(lemmas.items()):
        token_string = " ".join(sorted(list(token_set)))
        f.write(f"{lemma}: {token_string}\n")

print(f"DOne!")

files.download(TOKENS_FILE)
files.download(LEMMAS_FILE)

Processing...
DOne!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task-3: Inverted Index & Boolean Search Engine

In [13]:
!pip install pymorphy3

import collections

In [14]:
INPUT_DIR = "pages"
INVERTED_INDEX_FILE = "inverted_index.txt"

morph = MorphAnalyzer()
PRECEDENCE = {'NOT': 3, 'AND': 2, 'OR': 1, '(': 0, ')': 0}

In [15]:
def get_lemma(word):
    return morph.parse(word.lower())[0].normal_form

def build_inverted_index(directory):
    index = collections.defaultdict(set)
    doc_ids = set()

    for filename in sorted(os.listdir(directory)):
        if filename.endswith(".html"):
            doc_id = int(re.search(r'\d+', filename).group())
            doc_ids.add(doc_id)

            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as f:
                text = f.read().lower()
                words = re.findall(r'[а-яё]+', text)
                for word in words:
                    lemma = get_lemma(word)
                    index[lemma].add(doc_id)
    return index, sorted(list(doc_ids))

def boolean_eval(operator, operands, all_docs):
    if operator == 'NOT':
        return set(all_docs) - operands[0]
    elif operator == 'AND':
        return operands[0] & operands[1]
    elif operator == 'OR':
        return operands[0] | operands[1]
    return set()

def search(query, index, all_docs):
    tokens = re.findall(r'\(|\)|AND|OR|NOT|[а-яё]+', query, re.IGNORECASE)

    output_queue = []
    operator_stack = []

    for token in tokens:
        token_upper = token.upper()
        if token_upper in PRECEDENCE:
            if token_upper == '(':
                operator_stack.append(token_upper)
            elif token_upper == ')':
                while operator_stack and operator_stack[-1] != '(':
                    output_queue.append(operator_stack.pop())
                operator_stack.pop() # remocing the '(' patr
            else:
                while (operator_stack and
                       PRECEDENCE[operator_stack[-1]] >= PRECEDENCE[token_upper]):
                    output_queue.append(operator_stack.pop())
                operator_stack.append(token_upper)
        else:
            lemma = get_lemma(token)
            output_queue.append(index.get(lemma, set()))

    while operator_stack:
        output_queue.append(operator_stack.pop())

    stack = []
    for item in output_queue:
        if isinstance(item, set):
            stack.append(item)
        else:
            if item == 'NOT':
                op1 = stack.pop()
                stack.append(boolean_eval(item, [op1], all_docs))
            else:
                op2 = stack.pop()
                op1 = stack.pop()
                stack.append(boolean_eval(item, [op1, op2], all_docs))

    return stack[0] if stack else set()

In [17]:
print("Building inverted index...")
inverted_index, all_doc_ids = build_inverted_index(INPUT_DIR)

with open(INVERTED_INDEX_FILE, "w", encoding="utf-8") as f:
    for lemma, docs in sorted(inverted_index.items()):
        doc_list = " ".join(map(str, sorted(list(docs))))
        f.write(f"{lemma}: {doc_list}\n")

print(f"Inverted index saved to {INVERTED_INDEX_FILE}")
files.download(INVERTED_INDEX_FILE)

# print("\n--- Boolean Search ---")
# print("Operators: AND, OR, NOT, ()")

# user_query = input("Enter your query: ")
# results = search(user_query, inverted_index, all_doc_ids)

# print(f"\nResults for '{user_query}':")
# if results:
#     print(f"Found in documents: {sorted(list(results))}")
# else:
#     print("No documents found.")

Building inverted index...
Inverted index saved to inverted_index.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task-4: Vector Space Model

In [18]:
import math
import shutil

In [19]:
print("Loading tokens.txt...")
with open("tokens.txt", "r", encoding="utf-8") as f:
    ALL_TOKENS = [line.strip() for line in f if line.strip()]

print("Loading lemmas.txt...")
LEMMAS_DICT = {}
with open("lemmas.txt", "r", encoding="utf-8") as f:
    for line in f:
        if ":" in line:
            lemma, variants = line.split(":", 1)
            LEMMAS_DICT[lemma.strip()] = variants.strip().split()

INPUT_DIR = "pages"
TERMS_OUT = "tf_idf_terms"
LEMMAS_OUT = "tf_idf_lemmas"

for folder in [TERMS_OUT, LEMMAS_OUT]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)


Loading tokens.txt...
Loading lemmas.txt...


In [20]:
def run_tfidf_analysis():
    doc_filenames = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".html")])
    N = len(doc_filenames)
    docs_word_lists = []

    token_df = {token: 0 for token in ALL_TOKENS}
    lemma_df = {lemma: 0 for lemma in LEMMAS_DICT}

    print(f"Step 1/3: Analyzing {N} documents for Global Frequencies...")
    for idx, filename in enumerate(doc_filenames):
        with open(os.path.join(INPUT_DIR, filename), 'r', encoding='utf-8') as f:
            soup = BeautifulSoup(f.read(), 'html.parser')
            for s in soup(["script", "style"]): s.decompose()
            text = soup.get_text().lower()
            words = re.findall(r'[а-яё]+', text)
            docs_word_lists.append(words)

            unique_words = set(words)
            for t in ALL_TOKENS:
                if t in unique_words: token_df[t] += 1
            for l, variants in LEMMAS_DICT.items():
                if any(v in unique_words for v in variants): lemma_df[l] += 1

    print("Step 2/3: Pre-calculating IDF values...")
    token_idf = {t: math.log(N / df) if df > 0 else 0 for t, df in token_df.items()}
    lemma_idf = {l: math.log(N / df) if df > 0 else 0 for l, df in lemma_df.items()}

    print("Step 3/3: Calculating TF and writing result files...")
    for i, words in enumerate(docs_word_lists):
        total_word_count = len(words)
        if total_word_count == 0: continue
        doc_base_name = doc_filenames[i].replace(".html", ".txt")
        unique_words_in_doc = set(words)

        with open(os.path.join(TERMS_OUT, doc_base_name), "w", encoding="utf-8") as f:
            for t in ALL_TOKENS:
                if t in unique_words_in_doc:
                    tf = words.count(t) / total_word_count
                    tfidf = tf * token_idf[t]
                    f.write(f"{t} {token_idf[t]:.6f} {tfidf:.6f}\n")

        with open(os.path.join(LEMMAS_OUT, doc_base_name), "w", encoding="utf-8") as f:
            for l, variants in LEMMAS_DICT.items():
                if any(v in unique_words_in_doc for v in variants):
                    sum_variants = sum(words.count(v) for v in variants)
                    tf = sum_variants / total_word_count
                    tfidf = tf * lemma_idf[l]
                    f.write(f"{l} {lemma_idf[l]:.6f} {tfidf:.6f}\n")

In [21]:
run_tfidf_analysis()

print("Finalizing: Creating ZIP archive...")
result_zip = "tema4_tfidf_results.zip"
with zipfile.ZipFile(result_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in [TERMS_OUT, LEMMAS_OUT]:
        for root, _, files_list in os.walk(folder):
            for file in files_list:
                z.write(os.path.join(root, file), os.path.join(folder, file))

print(f"Success! ZIP size: {os.path.getsize(result_zip) / (1024*1024):.2f} MB")
files.download(result_zip)

Step 1/3: Analyzing 100 documents for Global Frequencies...
Step 2/3: Pre-calculating IDF values...
Step 3/3: Calculating TF and writing result files...
Finalizing: Creating ZIP archive...
Success! ZIP size: 1.13 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Task-5: Search Engine

In [22]:
morph = MorphAnalyzer()

TFIDF_DIR = "tf_idf_lemmas"
doc_vectors = {}
all_lemmas = set()

filenames = sorted([f for f in os.listdir(TFIDF_DIR) if f.endswith(".txt")])
for filename in filenames:
    doc_id = int(re.search(r'\d+', filename).group())
    doc_vectors[doc_id] = {}
    with open(os.path.join(TFIDF_DIR, filename), "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 3:
                lemma = parts[0].replace(":", "")
                tfidf = float(parts[2])
                doc_vectors[doc_id][lemma] = tfidf
                all_lemmas.add(lemma)

In [23]:
def get_query_to_vector(query):
    words = re.findall(r'[а-яё]+', query.lower())
    query_counts = collections.Counter([morph.parse(w)[0].normal_form for w in words])

    vector = {lemma: count for lemma, count in query_counts.items() if lemma in all_lemmas}
    return vector

def calculate_cosine_similarity(vec1, vec2):
    intersection = set(vec1.keys()) & set(vec2.keys())

    dot_product = sum(vec1[x] * vec2[x] for x in intersection)
    sum1 = sum(v**2 for v in vec1.values())
    sum2 = sum(v**2 for v in vec2.values())

    magnitude = math.sqrt(sum1) * math.sqrt(sum2)

    if not magnitude:
        return 0.0
    return dot_product / magnitude

In [24]:
def vector_search(query):
    query_vec = get_query_to_vector(query)
    if not query_vec:
        return []

    scores = []
    for doc_id, doc_vec in doc_vectors.items():
        score = calculate_cosine_similarity(query_vec, doc_vec)
        if score > 0:
            scores.append((doc_id, score))

    return sorted(scores, key=lambda x: x[1], reverse=True)

user_query = input("Enter search query: ")

results = vector_search(user_query)

if not results:
    print("No relevant documents found.")
else:
    print(f"\nTop results for '{user_query}':")
    print(f"{'Doc ID':<10} | {'Relevance Score':<15}")
    print("-" * 30)
    for doc_id, score in results[:10]:
        print(f"{doc_id:<10} | {score:.6f}")

Enter search query: оценка релевантности запроса

Top results for 'оценка релевантности запроса':
Doc ID     | Relevance Score
------------------------------
66         | 0.502287
54         | 0.256950
0          | 0.184761
71         | 0.173776
23         | 0.101343
15         | 0.084922
58         | 0.082796
26         | 0.077866
91         | 0.072519
9          | 0.057521
